# 58. 놓친 RG3 가스11개 — 압력전환·계량회복·온도형상 변화
다수결57번6TP/6FP를 고정한 채 다른 변화 방향6개씩 세 가설을 시험한다. RF·ExtraTrees각3seed평균과로지스틱3설정평균앙상블. 학습내부 구간제외 정상으로점수/변화확인경계를계산한다.
외부단위는실제Run확정값아닌5개라벨관측구간. 내부가드는원래RF기준이며전체57앵커완전중첩선택과는다르다. 기존테스트에서누락을보고선정한추가탐색으로독립검증을주장하지않는다. 최고점만보고모델을검증되었다고고르지않는다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
source=''.join(json.loads((root/'notebooks/54_rg3_pdf_thermal_confirm_gates.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(source.split('records=[]')[0])
F={
 'pressure_transition':pd.DataFrame({
 'pressure_minus_switch_delta':d.Max_Injection_Pressure__delta1-d.Max_Switch_Over_Pressure__delta1,
 'speed_delta':d.Max_Injection_Speed__delta1,
 'filling_delta':d.Filling_Time__delta1,
 'pressure_minus_switch_dev10':d.Max_Injection_Pressure__dev10-d.Max_Switch_Over_Pressure__dev10,
 'cushion_delta':d.Cushion_Position__delta1,
 'barrel_mold_delta_gap':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2}),
 'recovery_disturbance':pd.DataFrame({
 'meter_position_delta':d.Plasticizing_Position__delta1,
 'plasticizing_time_delta':d.Plasticizing_Time__delta1,
 'cycle_delta':d.Cycle_Time__delta1,
 'back_pressure_delta':d.Average_Back_Pressure__delta1,
 'cushion_dev10':d.Cushion_Position__dev10,
 'hopper_delta':d.Hopper_Temperature__delta1}),
 'thermal_shape_innovation':pd.DataFrame({
 'barrel2_minus3_delta':d.Barrel_Temperature_2__delta1-d.Barrel_Temperature_3__delta1,
 'barrel3_minus4_delta':d.Barrel_Temperature_3__delta1-d.Barrel_Temperature_4__delta1,
 'barrel4_minus5_delta':d.Barrel_Temperature_4__delta1-d.Barrel_Temperature_5__delta1,
 'barrel5_minus6_delta':d.Barrel_Temperature_5__delta1-d.Barrel_Temperature_6__delta1,
 'hopper_delta':d.Hopper_Temperature__delta1,
 'mold_delta_mean':(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2})
}
controls={
 'pressure_transition':['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3'],
 'recovery_disturbance':['Plasticizing_Position','Average_Back_Pressure','Hopper_Temperature','Barrel_Temperature_5'],
 'thermal_shape_innovation':['Barrel_Temperature_5','Hopper_Temperature','Mold_Temperature_3','Max_Injection_Speed']}
for name,x in F.items():x.assign(TimeStamp=d.TimeStamp).to_csv(tables/f'rg3_missed11_{name}_features.csv',index=False,encoding='utf-8-sig')
anchor=pd.read_csv(tables/'rg3_pdf_fixed_majority_predictions.csv',parse_dates=['TimeStamp']).set_index(['scheme','TimeStamp'])
memo={}
records=[];choices=[];innerrows=[];audits=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
 for j,b in enumerate(blocks):
  q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
  if len(np.unique(y[ref]))<2:continue
  if scheme=='past_blocks_only':assert d.TimeStamp.iloc[ref].max()<d.TimeStamp.iloc[q].min()
  anc=anchor.loc[(scheme,d.TimeStamp.iloc[q]),'prediction'].to_numpy(bool)
  ic=oldinner[(oldinner.scheme==scheme)&(oldinner.heldout_block==b)].set_index('TimeStamp').loc[d.TimeStamp.iloc[ref]]
  cb=float(np.quantile(ic.loc[y[ref]==0,'raw_baseline_score'],.99,method='higher'))
  oldpr=ic.raw_baseline_score.gt(cb).to_numpy()
  tn0,fp0,fn0,tp0=confusion_matrix(y[ref],oldpr,labels=[0,1]).ravel()
  f0=2*tp0/(2*tp0+fp0+fn0) if 2*tp0+fp0+fn0 else 0
  candidates=[]
  for name in F:
   outer=predict(ref,q,name);zo=local_residual(ref,q,name)
   inner=np.full((len(ref),4),np.nan);zi=np.full((len(ref),6),np.nan)
   for ib in groups.iloc[ref].unique():
    mask=groups.iloc[ref].eq(ib).to_numpy()
    inner[mask]=predict(ref[~mask],ref[mask],name)
    zi[mask]=local_residual(ref[~mask],ref[mask],name)
   novelty_i=np.max(np.abs(zi),axis=1);novelty_o=np.max(np.abs(zo),axis=1)
   novelty_cut=float(np.quantile(novelty_i[y[ref]==0],.95))
   for model,k in [('tree_linear_mean',1),('tree_linear_geometric',2)]:
    ns=(y[ref]==0)&np.isfinite(inner[:,k])
    for rate in [.005,.01,.025]:
     cut=float(np.quantile(inner[ns,k],1-rate,method='higher'))
     for gate in ['no_confirm','conditional_novelty95']:
      gi=np.ones(len(ref),bool) if gate=='no_confirm' else novelty_i>novelty_cut
      go=np.ones(len(q),bool) if gate=='no_confirm' else novelty_o>novelty_cut
      pr=oldpr|((inner[:,k]>cut)&gi)
      tn,fp,fn,tp=confusion_matrix(y[ref],pr,labels=[0,1]).ravel()
      fi=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
      row=dict(scheme=scheme,block=b,feature_group=name,model=model,budget=rate,gate=gate,inner_added_TP=int(tp-tp0),inner_added_FP=int(fp-fp0),inner_F1=fi,allowed=bool(tp>tp0 and fp-fp0<=2 and fi>f0),threshold=cut,novelty_threshold=novelty_cut)
      innerrows.append(row)
      extra=(outer[:,k]>cut)&go;candidates.append((row,extra))
      tag=name+'__'+model+'__'+gate
      for l,i in enumerate(q):records.append(dict(scheme=scheme,model=tag,budget=rate,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),prediction=int(anc[l]|extra[l]),anchor_prediction=int(anc[l]),score=float(outer[l,k]),threshold=cut,novelty=float(novelty_o[l]),novelty_threshold=novelty_cut))
  allowed=[c for c in candidates if c[0]['allowed']]
  best=max(allowed,key=lambda c:(c[0]['inner_F1'],-c[0]['inner_added_FP'])) if allowed else None
  pr=anc if best is None else anc|best[1]
  choices.append(dict(scheme=scheme,block=b,selected='none' if best is None else best[0]['feature_group']+'__'+best[0]['model']+'__'+best[0]['gate'],budget=np.nan if best is None else best[0]['budget']))
  for l,i in enumerate(q):records.append(dict(scheme=scheme,model='inner_guarded_selector',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),prediction=int(pr[l]),anchor_prediction=int(anc[l]),score=np.nan,threshold=np.nan,novelty=np.nan,novelty_threshold=np.nan))
  audits.append(dict(scheme=scheme,block=b,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum()),timestamp_overlap=len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q]))))
  print('Completed missed11 new signals',scheme,b,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,model,rate),g in pp.groupby(['scheme','model','budget'],sort=False):
 assert not g.TimeStamp.duplicated().any()
 assert ((g.anchor_prediction==0)|(g.prediction==1)).all()
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(scheme=scheme,model=model,budget=rate,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
mm=pd.DataFrame(rows)
for obj,name in [(mm,'metrics'),(pp,'predictions'),(pd.DataFrame(choices),'choices'),(pd.DataFrame(innerrows),'inner_candidates'),(pd.DataFrame(audits),'fold_audit')]:
 obj.to_csv(tables/f'rg3_missed11_dynamic_{name}.csv',index=False,encoding='utf-8-sig')
print('BEST EXPLORATORY ONLY',mm.sort_values(['scheme','F1','FP'],ascending=[True,False,True]).groupby('scheme').head(6).to_string(index=False))
print('INNER SELECTOR',mm[mm.model.eq('inner_guarded_selector')].to_string(index=False))
